In [1]:
from unittest import result

import pandas as pd

In [2]:
ground_truth = pd.read_csv('data/ground_truth-new.csv')

In [3]:
ground_truth = ground_truth.to_dict(orient='records')

In [4]:
ground_truth[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [5]:
from Module01_AgenticRAG.ingest import load_faq_data, built_index

documents =  load_faq_data()
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
len(documents_llm)

115

In [6]:
index = built_index(documents_llm)

In [7]:
results = index.search(ground_truth[0]['question'],
             num_results=5)

In [8]:
for r in results:
    print(f'{r['id']} == {ground_truth[0]['document']}: {ground_truth[0]['document'] == r['id']}')

74eb249bbf == 74eb249bbf: True
977bf7786c == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
489dd1c9d9 == 74eb249bbf: False
cdc3b285e5 == 74eb249bbf: False


In [9]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = index.search(q['question'],
                           num_results=5)
    relevance = []
    for d in results:
        relevance.append(int(d['id']==doc_id))
    return relevance

In [10]:
compute_relevance_text(ground_truth[0])

[1, 0, 0, 0, 0]

In [11]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [12]:
compute_relevance_total_text(ground_truth)

  0%|          | 0/395 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0,

In [13]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [14]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [15]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)


  0%|          | 0/395 [00:00<?, ?it/s]

In [18]:
sample = relevance_total[:15]

In [19]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt+=1
    return cnt/len(relevance)

In [20]:
hit_rate(relevance_total)

0.6506329113924051

In [21]:
import numpy as np
def mean_reciprocal_rank(relevance):
    df = pd.DataFrame(relevance)
    first_hit_idx = df.idxmax(axis=1)
    has_hit = df.max(axis=1)>=1
    reciprocal_ranks = np.where(has_hit,1.0/(first_hit_idx+1),0.0)
    return reciprocal_ranks.mean()


In [22]:
mean_reciprocal_rank(relevance_total)

np.float64(0.5756118143459916)

In [25]:
def evaluate(ground_truth,text_search):
    relevance_total = compute_relevance_total(ground_truth, text_search)
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mean_reciprocal_rank(relevance_total),
    }

In [26]:
evaluate(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': np.float64(0.5756118143459916)}